# 03 — Data Preprocessing

## 1. Objective

The objective of this notebook is to prepare the UCI Cleveland Heart Disease dataset for Machine Learning model development while maintaining a reproducible and leakage-free preprocessing workflow.

Based on the findings from the Exploratory Data Analysis (EDA), this notebook will:

- Separate the predictor variables (X) from the target variable (y).

- Convert the original heart-disease target variable into a binary classification target, where 0 represents no heart disease and 1 represents the presence of heart disease.

- Identify numerical and categorical features based on their data characteristics and meaning.

- Split the dataset into training and testing subsets using an 80:20 stratified split with random_state=42.

- Handle missing values using appropriate imputation strategies:

- Median imputation for numerical features.

- Most-frequent imputation for categorical features.

- Standardize numerical features using StandardScaler.

- Convert categorical features into numerical representations using OneHotEncoder.

- Build a reusable ColumnTransformer to apply the appropriate preprocessing steps to each feature type.

- Fit the preprocessing transformer using training data only to prevent data leakage.

- Verify the dimensions and structure of the transformed training and testing datasets.

- Prepare the preprocessing transformer for integration into Machine Learning pipelines and stratified cross-validation in subsequent notebooks.

The final output of this notebook will be a validated preprocessing pipeline that can be reused consistently across all baseline and advanced Machine Learning models developed in subsequent experiments.

## 2. Import Required Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

print("All imports successful!")

All imports successful!


## 3. Load Dataset

In [2]:
DATA_PATH = Path("../data/raw/processed.cleveland.data")

print("Dataset path:", DATA_PATH.resolve())
print("Dataset exists:", DATA_PATH.exists())

columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "num"
]

df = pd.read_csv(
    DATA_PATH,
    names=columns,
    na_values="?"
)

print("Dataset shape:", df.shape)

display(df.head())
df.info()

print("Missing values:")
print(df.isnull().sum())

Dataset path: ../data/raw/processed.cleveland.data
Dataset exists: True
Dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB
Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: 

## 4. Define Features and Target

In [3]:
predictor_features = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
]

X = df[predictor_features].copy()
y = df["num"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("Original target distribution:")
print(y.value_counts().sort_index())


X shape: (303, 13)
y shape: (303,)
Original target distribution:
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


## 5. Convert Target to Binary Classification

In [4]:
y = (y > 0).astype(int)

print("Binary target distribution:")
print(y.value_counts().sort_index())

print("Binary target percentages:")
print(y.value_counts(normalize=True).sort_index() * 100)

Binary target distribution:
num
0    164
1    139
Name: count, dtype: int64
Binary target percentages:
num
0    54.125413
1    45.874587
Name: proportion, dtype: float64


### 5.1 Target and Leakage Validation

The original `num` variable is used only to construct the binary target and is not included among the predictors. No target-derived feature is created. The checks below confirm the expected target domain and predictor count before the train/test split.


In [5]:
assert "num" not in predictor_features
assert len(predictor_features) == 13
assert set(y.unique()) <= {0, 1}
assert len(X) == len(y) == 303

print("Predictor count:", X.shape[1])
print("Target values:", sorted(y.unique().tolist()))
print("Target leakage check: PASSED")


Predictor count: 13
Target values: [0, 1]
Target leakage check: PASSED


## 6. Identify Numerical and Categorical Features

In [6]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

print("Number of predictors:", len(predictor_features))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

assert set(numerical_features + categorical_features) == set(predictor_features)

Number of predictors: 13
Numerical features: 5
Categorical features: 8


### 6.1 Missing-Value Summary

The working Cleveland file contains six missing predictor cells: four in `ca` and two in `thal`. Because the missingness mechanism cannot be established from this descriptive check alone, the project does not classify it as MCAR, MAR, or MNAR. No observations are deleted; the missing values are handled inside the training-fold preprocessing pipelines.


In [7]:
missing_summary = X.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

print("Missing predictor cells:", int(X.isna().sum().sum()))
print("Overall missing predictor-cell percentage: {:.2f}%".format(100 * X.isna().sum().sum() / (len(X) * X.shape[1])))
display(missing_summary.to_frame("Missing"))


Missing predictor cells: 6
Overall missing predictor-cell percentage: 0.15%


,Missing
ca,4
thal,2


## 7. Train/Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (242, 13)
X_test shape: (61, 13)
y_train shape: (242,)
y_test shape: (61,)

Training target distribution:
num
0    0.541322
1    0.458678
Name: proportion, dtype: float64

Testing target distribution:
num
0    0.540984
1    0.459016
Name: proportion, dtype: float64


## 8. Define Numerical Preprocessing Pipeline

In [9]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

## 9. Define Categorical Preprocessing Pipeline

In [10]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

## 10. Build ColumnTransformer

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)


## 11. Fit Preprocessor Using Training Data

In [12]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 12. Transform Training and Testing Data

In [13]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

display(X_train_processed_df.head())

Processed X_train shape: (242, 28)
Processed X_test shape: (61, 28)


,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0.0,cat__sex_1.0,cat__cp_1.0,cat__cp_2.0,cat__cp_3.0,cat__cp_4.0,cat__fbs_0.0,cat__fbs_1.0,cat__restecg_0.0,cat__restecg_1.0,cat__restecg_2.0,cat__exang_0.0,cat__exang_1.0,cat__slope_1.0,cat__slope_2.0,cat__slope_3.0,cat__ca_0.0,cat__ca_1.0,cat__ca_2.0,cat__ca_3.0,cat__thal_3.0,cat__thal_6.0,cat__thal_7.0
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 13. Verify Processed Dataset

### 13.1 Check Missing Values

After imputation and encoding, the transformed training and testing matrices should contain no missing values.


In [14]:
print(
    "Missing values in processed training data:",
    np.isnan(X_train_processed.toarray()).sum()
    if hasattr(X_train_processed, "toarray")
    else np.isnan(X_train_processed).sum()
)

print(
    "Missing values in processed test data:",
    np.isnan(X_test_processed.toarray()).sum()
    if hasattr(X_test_processed, "toarray")
    else np.isnan(X_test_processed).sum()
)

Missing values in processed training data: 0
Missing values in processed test data: 0


### 13.2 Get Processed Feature Names

The five numerical variables remain numerical after scaling. The eight categorical variables are one-hot encoded. For the current Cleveland data, this produces 28 transformed features.


In [15]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for i, name in enumerate(feature_names, start=1):
    print(i, name)

print("Original number of features:", X.shape[1])
print("Processed number of features:", X_train_processed.shape[1])

Number of processed features: 28
1 num__age
2 num__trestbps
3 num__chol
4 num__thalach
5 num__oldpeak
6 cat__sex_0.0
7 cat__sex_1.0
8 cat__cp_1.0
9 cat__cp_2.0
10 cat__cp_3.0
11 cat__cp_4.0
12 cat__fbs_0.0
13 cat__fbs_1.0
14 cat__restecg_0.0
15 cat__restecg_1.0
16 cat__restecg_2.0
17 cat__exang_0.0
18 cat__exang_1.0
19 cat__slope_1.0
20 cat__slope_2.0
21 cat__slope_3.0
22 cat__ca_0.0
23 cat__ca_1.0
24 cat__ca_2.0
25 cat__ca_3.0
26 cat__thal_3.0
27 cat__thal_6.0
28 cat__thal_7.0
Original number of features: 13
Processed number of features: 28


### 13.3 Convert Processed Training Data to DataFrame

In [16]:
X_train_processed_dense = (
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed
)

processed_df = pd.DataFrame(
    X_train_processed_dense,
    columns=feature_names,
    index=X_train.index
)

display(processed_df.head())

,num__age,num__trestbps,num__chol,num__thalach,num__oldpeak,cat__sex_0.0,cat__sex_1.0,cat__cp_1.0,cat__cp_2.0,cat__cp_3.0,cat__cp_4.0,cat__fbs_0.0,cat__fbs_1.0,cat__restecg_0.0,cat__restecg_1.0,cat__restecg_2.0,cat__exang_0.0,cat__exang_1.0,cat__slope_1.0,cat__slope_2.0,cat__slope_3.0,cat__ca_0.0,cat__ca_1.0,cat__ca_2.0,cat__ca_3.0,cat__thal_3.0,cat__thal_6.0,cat__thal_7.0
180,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
208,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
167,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
105,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
297,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


### 13.4 Inspect Processed Dataset

In [17]:
print("Processed dataset shape:", processed_df.shape)

display(processed_df.info())

print("Missing values in processed dataset:")
print(processed_df.isnull().sum().sum())


Processed dataset shape: (242, 28)
<class 'pandas.core.frame.DataFrame'>
Index: 242 entries, 180 to 16
Data columns (total 28 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   num__age          242 non-null    float64
 1   num__trestbps     242 non-null    float64
 2   num__chol         242 non-null    float64
 3   num__thalach      242 non-null    float64
 4   num__oldpeak      242 non-null    float64
 5   cat__sex_0.0      242 non-null    float64
 6   cat__sex_1.0      242 non-null    float64
 7   cat__cp_1.0       242 non-null    float64
 8   cat__cp_2.0       242 non-null    float64
 9   cat__cp_3.0       242 non-null    float64
 10  cat__cp_4.0       242 non-null    float64
 11  cat__fbs_0.0      242 non-null    float64
 12  cat__fbs_1.0      242 non-null    float64
 13  cat__restecg_0.0  242 non-null    float64
 14  cat__restecg_1.0  242 non-null    float64
 15  cat__restecg_2.0  242 non-null    float64
 16  cat__exang_0.

None

Missing values in processed dataset:
0


## 14. Leakage Prevention and Pipeline Check

The preprocessing transformer is fitted on the training subset for the transformation checks above. For model development, the same transformation design is embedded inside each model pipeline and therefore fitted independently within each training fold.


In [18]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

check_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

print("Pipeline integration check:", check_pipeline.named_steps)
print("Preprocessing is embedded in the model pipeline and will be fitted within each training fold during cross-validation.")


Pipeline integration check: {'preprocessor': ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalach',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sex', 'cp', 'fbs', 'restecg', 'exang',
                                  'slope', 'ca', 'thal'])]), 'mo

## 15. Preprocessing Conclusions

The Cleveland heart disease dataset contains 303 observations and 13 predictors. After conversion of the original `num` variable, the binary target contains 164 observations without disease and 139 with disease. The dataset is divided into training and testing subsets using an 80:20 stratified split with `random_state = 42`, producing 242 training observations and 61 test observations.

The preprocessing strategy follows the feature types identified during EDA. The five numerical features (`age`, `trestbps`, `chol`, `thalach`, and `oldpeak`) are processed using median imputation followed by standardization. The eight categorical features (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, and `thal`) are processed using most-frequent imputation followed by one-hot encoding with unknown categories ignored. The `ca` and `thal` variables are treated as categorical because their values represent discrete clinical categories rather than continuous measurements.

The working dataset contains six missing predictor cells: four in `ca` and two in `thal`, corresponding to 0.15% of all predictor cells. No rows are removed because of these values. The missingness mechanism is not inferred from this descriptive analysis.

The preprocessing transformer is fitted on training data only when this notebook demonstrates the transformation. For model development, the same transformation design is embedded directly inside each model pipeline. Consequently, imputation, scaling, and one-hot encoding are learned only from the training portion of each cross-validation fold. This prevents information from validation folds or the final test set from entering model training.

The transformed data contains 28 features: five scaled numerical features and 23 one-hot encoded categorical levels. The final test set remains untouched for model development and selection and is reserved for the final evaluation of the selected baseline model. Overall, this notebook establishes a reproducible, leakage-aware preprocessing specification for subsequent baseline and advanced modelling.
